# Rasterio Walkthrough

#### import and read in

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
dataset = rasterio.open(r"deu_pop_2025_CN_100m_R2025A_v1.tif")

#### Dataset Properties

In [2]:
dataset.name

'deu_pop_2025_CN_100m_R2025A_v1.tif'

In [3]:
dataset.count

1

In [4]:
print(f"The dataset has {dataset.height} rows and {dataset.width} columns.")

The dataset has 9347 rows and 11010 columns.


In [5]:
{datatype for datatype in zip(dataset.dtypes, dataset.indexes)}

{('float32', 1)}

#### Georeferencing

In [6]:
print(dataset.bounds)

BoundingBox(left=5.866665923199998, bottom=47.27000014691999, right=15.041665886499997, top=55.05916678242999)


Top and bottom measure latitude, left and right longitude. One degree is relative to your position. This bbox enclosers Germany.

Roughly to Germany:
- 1° latitude: ca. 111 km
- 1° longitude: ca. 68 km

In [7]:
one_degrees_ns = 111
one_degrees_ew = 68
east = 15.04
west = 5.86
north = 55.05
south = 47.27

print(f"The east-west extent is {round(east - west, 2)} degrees, which is approximately {round(one_degrees_ew * (east - west), 2)} km.")
print(f"The north-south extent is {round(north - south, 2)} degrees, which is approximately {round(one_degrees_ns * (north - south), 2)} km.")
print(f"The total area of ​​the dataset is approximately {int((one_degrees_ew * (east - west)) * (one_degrees_ns * (north - south)))} km².")

The east-west extent is 9.18 degrees, which is approximately 624.24 km.
The north-south extent is 7.78 degrees, which is approximately 863.58 km.
The total area of ​​the dataset is approximately 539081 km².


In [8]:
print(dataset.crs)

EPSG:4326


#### Convert systems

To perform spatial calculations in meters rather than degrees (WGS84), the data must be reprojected into a metric Coordinate Reference System (CRS) using warping. This process maps the original pixel values onto a new raster grid.

- Target CRS: ETRS89 / UTM Zone 33N

- EPSG Code: EPSG:25833

In [ ]:
print(dataset.crs)

ziel_crs = "EPSG:25833"

with rasterio.open(dataset.name) as source:
    transform, width, height = calculate_default_transform(source.crs, ziel_crs, source.width, source.height, *source.bounds)
    kwargs = source.meta.copy()
    kwargs.update({
        'crs': ziel_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open("2025_population_in_meters", 'w', **kwargs) as dst:
        for i in range(1, source.count + 1):
            reproject(
                source=rasterio.band(source, i),
                destination=rasterio.band(dst, i),
                source_transform=source.transform,
                source_crs=source.crs,
                dst_transform=transform,
                dst_crs=ziel_crs,
                resampling=Resampling.nearest
            )
meters_dataset = rasterio.open("2025_population_in_meters")
print(meters_dataset.crs)

EPSG:4326
EPSG:25833


In [10]:
dataset.close()

For a population data set, this is completely useless but it is important to know when to work with tif that has calculations.